# Pilot Skripsi di Google Colab
**Rahul Ghulam Hidayatulloh — Klasifikasi Kondisi Sel CD4 pada Infeksi HIV (ekstraksi fitur + machine learning)**

Notebook ini menjalankan pipeline yang sama dengan laptop (`docs/PEMROSESAN-DATA.md`), tetapi di Colab:
1. Hubungkan Google Drive (data mentah ada di `MyDrive/Project-Skripsi/data/raw/`).
2. Ambil kode dari repo GitHub privat `raaahl15/Project-Skripsi`.
3. Ekstrak data ke disk Colab, jalankan heiDATA & CyTOF Luo (opsional: Essig).
4. Simpan hasil ke `MyDrive/Project-Skripsi/results-colab/<waktu>/`.

**Persiapan sekali saja — token GitHub (jangan ditulis di notebook):**
- GitHub → Settings → Developer settings → *Fine-grained tokens* → buat token dengan akses **Contents: Read-only** hanya untuk repo `Project-Skripsi`.
- Di Colab: ikon **kunci (Secrets)** di panel kiri → *Add new secret* → nama `GITHUB_TOKEN`, isi token → aktifkan *Notebook access*.

Jalankan sel dari atas ke bawah (**Runtime → Run all**). Runtime CPU sudah cukup untuk ekstraksi fitur + ML.

## 1. Spesifikasi runtime

In [ ]:
!nproc
!free -h | head -2
!df -h /content | tail -1
import torch; print('GPU:', torch.cuda.is_available())

## 2. Hubungkan Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Project-Skripsi'
!ls -la "$DRIVE/data/raw"

## 3. Ambil kode dari GitHub (token dari Colab Secrets)

In [ ]:
import os, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO = '/content/skripsi-ml'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', f'https://{token}@github.com/raaahl15/Project-Skripsi.git', REPO], check=True)
    # remove the token from the saved remote URL
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', 'https://github.com/raaahl15/Project-Skripsi.git'], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', f'https://{token}@github.com/raaahl15/Project-Skripsi.git'], check=True)
del token
%cd /content/skripsi-ml
!git log --oneline -1

## 4. Pasang paket yang belum ada di Colab

In [ ]:
!pip install -q flowio shap tifffile h5py

## 5. Siapkan data (ekstrak dari Drive ke disk Colab)

In [ ]:
import os, zipfile
RAW = '/content/skripsi-ml/data/raw'
os.makedirs(RAW, exist_ok=True)
jobs = {
    'heidata-MUXYRY': f'{DRIVE}/data/raw/dataverse_files.zip',               # heiDATA (Ananth et al. 2024)
    'dryad-luo-2023': f'{DRIVE}/data/raw/doi_10_7272_Q6SX6BFR__v20230217.zip', # CyTOF (Luo et al. 2022)
}
for name, zp in jobs.items():
    dst = f'{RAW}/{name}'
    if os.path.isdir(dst) and os.listdir(dst):
        print('sudah ada:', dst); continue
    if not os.path.exists(zp):
        print('TIDAK DITEMUKAN di Drive:', zp); continue
    print('ekstrak', zp, '->', dst)
    zipfile.ZipFile(zp).extractall(dst)
!du -sh data/raw/*

## 6. heiDATA — segmentasi, ekstraksi fitur, klasifikasi

In [ ]:
!python src/pilot_heidata_features.py --seg classical
!python src/pilot_heidata_classify.py --n-perm 100

## 7. CyTOF Luo — inventaris, gating, klasifikasi

In [ ]:
!python src/pilot_luo_inspect.py
!python src/pilot_luo_classify.py

## 8. (Opsional) IFC Essig — pilot CNN
Tidak menjadi prioritas (bimbingan 4: *machine learning*, bukan *deep learning*). Aktifkan hanya bila diminta pembimbing.
Butuh `exp1.zip` (19,6 GB) di `MyDrive/Project-Skripsi/data/raw/`. Disarankan *Runtime → Change runtime type → GPU*.

In [ ]:
RUN_ESSIG_CNN = False
ESSIG_ZIP = f'{DRIVE}/data/raw/exp1.zip'
if RUN_ESSIG_CNN:
    local_zip = '/content/exp1.zip'
    if not os.path.exists(local_zip):
        !cp "$ESSIG_ZIP" /content/   # copy to local disk: much faster random access than the Drive mount
    !python src/pilot_cnn_transfer.py --essig-zip /content/exp1.zip
else:
    print('Lewati Essig (RUN_ESSIG_CNN = False)')

## 9. Simpan hasil ke Google Drive

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
dst = f'{DRIVE}/results-colab/{stamp}'
shutil.copytree('/content/skripsi-ml/results', dst, dirs_exist_ok=True)
print('Hasil disimpan di:', dst)
!ls "$dst"

## 10. Setelah selesai
- Hasil ada di Drive `Project-Skripsi/results-colab/<waktu>/`. Di laptop, salin ke repo:
  `rclone copy "gdrive:Project-Skripsi/results-colab/<waktu>" results` lalu commit.
- Catat run ini di `docs/RIWAYAT-PENGERJAAN.md` dan Work Order terkait.